## Guided modes of an anisotropic waveguide

Lithium niobate, liquid crystals and many other optical materials are anisotropic: the refractive index that light sees depends on its polarization. Build a waveguide out of such a material and the anisotropy splits modes that would otherwise travel at the same speed, and makes their effective index depend on how the crystal is cut. This notebook computes the guided modes of a square waveguide with an anisotropic core, using the finite element package [gyptis](https://gyptis.gitlab.io), for three orientations of the crystal, and checks the results against a classic benchmark ([Koshiba, Hayata and Suzuki, *J. Lightwave Technol.* **4**, 121 (1986)](https://doi.org/10.1109/JLT.1986.1074687)).

Along the way we meet the main ingredients of a modal simulation: the effective index, the guiding condition, vector edge elements, an eigenvalue problem for the propagation constant, and the polarization of the modes.

### Guided modes and effective index

A waveguide is invariant along its axis $z$. We look for fields that keep their shape as they propagate,

$$
\mathbf{E}(x, y, z) = \mathbf{E}(x, y)\, e^{-j\beta z},
$$

with the $e^{j\omega t}$ time convention used by gyptis. The propagation constant $\beta$ tells how fast the phase advances along the guide. Comparing it with the vacuum wavenumber $k_0 = 2\pi/\lambda$ defines the **effective index**

$$
n_\text{eff} = \frac{\beta}{k_0}:
$$

the mode travels as a plane wave would in a uniform medium of index $n_\text{eff}$. Substituting this field into Maxwell's equations (non-magnetic media, $\mu = 1$) gives

$$
\nabla\times\nabla\times\mathbf{E} - k_0^2\,\varepsilon\,\mathbf{E} = 0,
\qquad
\nabla = \nabla_t - j\beta\,\hat{\mathbf{z}},
$$

where $\nabla_t$ acts on $x$ and $y$ only. For a given frequency, this is an eigenvalue problem for $\beta$. Its solutions are a few discrete **guided modes**, plus a continuum of radiation modes that leak into the cladding. A mode is guided when

$$
n_\text{clad} < n_\text{eff} < n_\text{core}:
$$

above $n_\text{clad}$ the field decays exponentially away from the core, so it carries its power along the guide without loss; and no mode can travel slower than light in the densest material it sees.

### A uniaxial core

The core is a uniaxial crystal. One direction, the optic axis $\hat{\mathbf{c}}$, is special: an electric field along $\hat{\mathbf{c}}$ sees the extraordinary permittivity $\varepsilon_e$, a field perpendicular to it the ordinary permittivity $\varepsilon_o$. As a tensor,

$$
\varepsilon = \varepsilon_o\,\mathbf{I} + (\varepsilon_e - \varepsilon_o)\,\hat{\mathbf{c}}\,\hat{\mathbf{c}}^{\mathsf{T}}.
$$

A plane wave travelling perpendicular to $\hat{\mathbf{c}}$ therefore has two indices: $n_o = \sqrt{\varepsilon_o}$ when polarized perpendicular to the axis, $n_e = \sqrt{\varepsilon_e}$ when polarized along it. This is birefringence.

We take the values of the benchmark: a square core of side $t$ with $\varepsilon_o = 2.31$ and $\varepsilon_e = 2.19$, buried in an isotropic cladding with $\varepsilon = 2.05$. Only the orientation of the optic axis changes:

- **along $z$**, the axis of the guide;
- **along $x$**, parallel to one side of the core;
- **diagonal**, along $(\hat{\mathbf{x}} + \hat{\mathbf{y}})/\sqrt{2}$ in the cross section.

All lengths are in units of $t$, so the frequency enters through the dimensionless product $k_0 t$.

In [ ]:
import urllib.request
from collections import OrderedDict

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.tri import LinearTriInterpolator, Triangulation

import gyptis as gy

plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": "medium"})

In [ ]:
t = 1                          # side of the core: the unit of length
eps_o, eps_e = 2.31, 2.19      # ordinary and extraordinary permittivities of the core
eps_clad = 2.05                # isotropic cladding


def uniaxial(c):
    # Permittivity tensor of the core for an optic axis along c.
    c = np.asarray(c, float) / np.linalg.norm(c)
    return eps_o * np.eye(3) + (eps_e - eps_o) * np.outer(c, c)


axes = {"along z": [0, 0, 1], "along x": [1, 0, 0], "diagonal": [1, 1, 0]}

print(f"n_o = {eps_o**0.5:.4f}, n_e = {eps_e**0.5:.4f}, n_clad = {eps_clad**0.5:.4f}\n")
for name, c in axes.items():
    print(f"optic axis {name}:\n{uniaxial(c).round(3)}\n")

The three tensors share the same eigenvalues, only their principal axes turn. With the axis along $z$, the cross section is isotropic: both transverse components see $\varepsilon_o$. With the axis along $x$, the tensor is diagonal but $E_x$ and $E_y$ see different permittivities. With the diagonal axis, an off-diagonal term $\varepsilon_{xy}$ couples $E_x$ and $E_y$.

### Finite element formulation

We split the field into its transverse part $\mathbf{E}_t = (E_x, E_y)$ and its longitudinal part $E_z$. They call for different finite elements.

- $\mathbf{E}_t$ uses **edge (Nédélec) elements**, whose degrees of freedom are tangential components along the triangle edges. They enforce continuity of the tangential field across element boundaries and nothing more, which is exactly what Maxwell's equations demand at a material interface, where the normal component jumps with $\varepsilon$. Nodal elements, which force all components to be continuous, produce spurious non-physical modes in this problem.
- $E_z$ is continuous everywhere and uses ordinary nodal (Lagrange) elements.

With the scaled unknowns $\mathbf{e}_t = \beta\,\mathbf{E}_t$ and $e_z = -j E_z$ ([Lee, Sun and Cendes, *IEEE Trans. Microw. Theory Tech.* **39**, 1262 (1991)](https://doi.org/10.1109/22.85399)), the weak form becomes a generalized eigenvalue problem that is linear in $\beta^2$: for all test functions $(\mathbf{v}_t, v_z)$,

$$
\int_\Omega \left[(\nabla_t\times\mathbf{e}_t)(\nabla_t\times\mathbf{v}_t) - k_0^2\,\varepsilon_{tt}\,\mathbf{e}_t\cdot\mathbf{v}_t\right]\mathrm{d}\Omega
= -\beta^2 \int_\Omega \left[(\mathbf{e}_t + \nabla_t e_z)\cdot(\mathbf{v}_t + \nabla_t v_z) - k_0^2\,\varepsilon_{zz}\,e_z\,v_z\right]\mathrm{d}\Omega,
$$

where $\varepsilon_{tt}$ is the $2\times2$ transverse block of the tensor. Both sides are sparse matrices. An iterative (Krylov–Schur) eigensolver with a shift-and-invert transform finds the eigenvalues closest to a chosen target; we aim just above $n_o k_0$, the largest $\beta$ a guided mode can have.

This form assumes a block-diagonal tensor ($\varepsilon_{xz} = \varepsilon_{yz} = 0$): with the scaling above, those coupling terms would bring odd powers of $\beta$ and a quadratic eigenproblem. It holds for our three cases, where the optic axis lies either in the cross section or along the guide.

### Geometry and mesh

gyptis builds the cross section as a layered box surrounded by perfectly matched layers (PMLs). We give both layers the cladding permittivity, so the core sits in a uniform cladding. Guided modes decay away from the core and barely reach the PMLs; radiation modes run into them and get absorbed, which turns their eigenvalues complex. That is how we will tell them apart.

The fundamental modes vary slowly across the core, so a moderate mesh suffices: triangles of size $t/10$ in the core and $t/3$ in the cladding. We check this choice at the end.

In [ ]:
L = 3                          # cladding around the core
pml = 2                        # thickness of the PMLs


def make_geometry(h_core, h_clad):
    geom = gy.geometry.LayeredBoxPML2D(
        t + 2 * L, thicknesses=OrderedDict(substrate=L, superstrate=L + t), pml_width=(pml, pml)
    )
    core = geom.add_rectangle(-t / 2, geom.y_position["superstrate"], 0, t, t)
    core, sup, sub = geom.fragment(core, [geom.layers["superstrate"], geom.layers["substrate"]])
    geom.add_physical(core, "core")
    geom.add_physical(sub, "substrate")
    geom.add_physical(sup, "superstrate")
    for p in geom.pmls:
        geom.set_size(p, 2 * h_clad)
    geom.set_size("substrate", h_clad)
    geom.set_size("superstrate", h_clad)
    geom.set_size("core", h_core)
    geom.build()
    return geom

In [ ]:
geom = make_geometry(t / 10, t / 3)
print(f"{geom.mesh.num_cells()} triangles")

fig, ax = plt.subplots(figsize=(4.4, 4.4))
geom.plot_mesh(lw=0.3)
geom.plot_subdomains(c="tab:red", lw=1)
ax.set_aspect("equal")
plt.axis("off")
plt.show()

### Solving for the modes

`guided_modes` sets up the problem for one orientation of the optic axis and one frequency, solves it, keeps the modes with a real effective index above $n_\text{clad}$, and sorts them from the most to the least confined. `polarization` measures which way a mode's transverse field points: the fraction of $\int|\mathbf{E}_t|^2$ carried by the component along an in-plane unit vector $\hat{\mathbf{u}}$, integrated over the physical domain (PMLs excluded). It assembles the overlaps $M_{ij} = \mathrm{Re}\int E_i E_j^*\,\mathrm{d}\Omega$ once, so that the fraction along any $\hat{\mathbf{u}}$ is $\hat{\mathbf{u}}^{\mathsf{T}} M \hat{\mathbf{u}} / \mathrm{tr}\,M$.

In [ ]:
def guided_modes(c, k0t, geom=geom, n_eig=8, degree=(1, 1)):
    k0 = k0t / t
    epsilon = dict(core=uniaxial(c), substrate=eps_clad, superstrate=eps_clad)
    simu = gy.Waveguide(geom, epsilon=epsilon, wavenumber=k0, degree=degree)
    simu.eigensolve(n_eig=n_eig, target=1.02 * k0 * eps_o**0.5, tol=1e-6, maximum_iterations=30)
    neff = np.array(simu.solution["eigenvalues"]) / k0
    guided = [(n.real, E) for n, E in zip(neff, simu.solution["eigenvectors"])
              if abs(n.imag) < 1e-4 and n.real > eps_clad**0.5]
    guided.sort(key=lambda m: -m[0])
    return np.array([n for n, _ in guided]), [E for _, E in guided]


dx = geom.measure["dx"]
dx_phys = dx("core") + dx("substrate") + dx("superstrate")


def polarization(E, u):
    # Overlaps M_ij = Re of the integral of E_i E_j^*, then u^T M u / trace(M).
    M = np.array([[gy.dolfin.assemble((E[i].real * E[j].real + E[i].imag * E[j].imag) * dx_phys)
                   for j in range(2)] for i in range(2)])
    u = np.asarray(u, float)[:2] / np.linalg.norm(u[:2])
    return u @ M @ u / np.trace(M)

#### Optic axis along the guide

We start at $k_0 t = 10$ with the optic axis along $z$. The cross section is then isotropic for transverse fields, and the square core is symmetric under a quarter turn, which maps a mode polarized along $x$ onto one polarized along $y$. The two fundamental modes, traditionally called $E^x_{11}$ and $E^y_{11}$ after their dominant field component, must share the same effective index.

In [ ]:
k0t = 10
neff_z, modes_z = guided_modes(axes["along z"], k0t)
for n, E in zip(neff_z, modes_z):
    print(f"n_eff = {n:.5f}   fraction of |E_t|^2 along x = {polarization(E, [1, 0]):.3f}")
print(f"\nsplitting of the fundamental pair: {neff_z[0] - neff_z[1]:.1e}")

The fundamental pair is degenerate up to a splitting of order $10^{-5}$, which comes from the mesh: it is unstructured and does not share the exact symmetry of the square. Any combination of two degenerate modes is also a mode, so in principle the pair has no preferred polarization; here the tiny asymmetry of the mesh happens to select modes polarized almost purely along $x$ and $y$. The next four modes form the second-order family, split into close pairs in the same way.

#### Optic axis in the cross section

Now we turn the optic axis into the cross section, first along $x$, then along the diagonal.

In [ ]:
modes_at_10 = {"along z": (neff_z, modes_z)}
for name in ("along x", "diagonal"):
    modes_at_10[name] = guided_modes(axes[name], k0t)

print(f"k0 t = {k0t}         fundamental pair         splitting")
for name, (neff, modes) in modes_at_10.items():
    print(f"optic axis {name:<9} {neff[0]:.5f}  {neff[1]:.5f}      {neff[0] - neff[1]:.5f}")
print(f"bulk crystal            n_o - n_e =          {eps_o**0.5 - eps_e**0.5:.5f}")

The degeneracy is lifted: the two fundamental modes now differ in effective index by a few $10^{-2}$, the same order as the bulk birefringence $n_o - n_e$, though smaller, because part of each mode travels in the isotropic cladding. The two in-plane orientations give the same pair to within $2\times10^{-5}$. The plots below show why: in both cases one mode is polarized perpendicular to the optic axis and sees $\varepsilon_o$, the other along it and sees $\varepsilon_e$. The orientation of the axis relative to the sides of the core hardly matters, and symmetry explains it. Treat the anisotropy as a small perturbation of an isotropic core: it shifts the effective index of a mode polarized along $\hat{\mathbf{u}}$ in proportion to the overlap of the mode with the core, weighted by $\hat{\mathbf{u}}^{\mathsf{T}}\hat{\mathbf{c}}$. The quarter-turn symmetry of the square makes that overlap the same for every in-plane direction, so to first order the core cannot tell a side from a diagonal. Only the vector corrections near the corners of the core, a weaker effect, separate the two cases.

To draw the transverse field, we project $E_x$ and $E_y$ onto nodal elements, remove the arbitrary global phase of the eigenvector, and plot $|\mathbf{E}_t|$ with arrows for its direction. The optic axis is marked in green (a dot when it points out of the page).

In [ ]:
V = gy.dolfin.FunctionSpace(geom.mesh, "CG", 1)
tri = Triangulation(*geom.mesh.coordinates().T, geom.mesh.cells())


def transverse_field(E):
    ex, ey = [gy.utils.project_iterative(E[j].real, V).compute_vertex_values()
              + 1j * gy.utils.project_iterative(E[j].imag, V).compute_vertex_values() for j in range(2)]
    i = np.argmax(abs(ex) ** 2 + abs(ey) ** 2)
    phase = np.angle(ex[i] if abs(ex[i]) > abs(ey[i]) else ey[i])
    ex, ey = (ex * np.exp(-1j * phase)).real, (ey * np.exp(-1j * phase)).real
    norm = np.hypot(ex, ey).max()
    return ex / norm, ey / norm


def plot_mode(ax, E, c, title):
    ex, ey = transverse_field(E)
    ax.tripcolor(tri, np.hypot(ex, ey), shading="gouraud", cmap="inferno", vmin=0, vmax=1)
    X, Y = np.meshgrid(np.linspace(-1.2, 1.2, 13), np.linspace(-1.2, 1.2, 13))
    U, W = (LinearTriInterpolator(tri, f)(X, Y) for f in (ex, ey))
    ax.quiver(X, Y, U, W, color="w", scale=12, width=0.006)
    ax.add_patch(plt.Rectangle((-t / 2, -t / 2), t, t, fill=False, ec="w", lw=0.8, ls="--"))
    if c[2]:
        ax.plot(1.25, 1.25, "o", mfc="none", mec="limegreen", ms=9, mew=1.5)
        ax.plot(1.25, 1.25, ".", c="limegreen", ms=5)
    else:
        u = np.array(c[:2], float) / np.linalg.norm(c[:2]) * 0.35
        ax.annotate("", xy=(1.25 + u[0], 1.25 + u[1]), xytext=(1.25 - u[0], 1.25 - u[1]),
                    arrowprops=dict(arrowstyle="<->", color="limegreen", lw=1.5))
    ax.set(xlim=(-1.7, 1.7), ylim=(-1.7, 1.7), aspect="equal", title=title)
    ax.axis("off")


fig, axs = plt.subplots(2, 3, figsize=(9, 6.2))
for col, (name, (neff, modes)) in enumerate(modes_at_10.items()):
    for row in range(2):
        plot_mode(axs[row, col], modes[row], axes[name],
                  f"axis {name}\n$n_\\mathrm{{eff}} = {neff[row]:.4f}$" if row == 0 else f"$n_\\mathrm{{eff}} = {neff[row]:.4f}$")
plt.tight_layout()
plt.show()

With the optic axis in the cross section, the mode with the higher index is polarized perpendicular to the axis (ordinary-like, it sees $\varepsilon_o = 2.31$), the one with the lower index along it (extraordinary-like, $\varepsilon_e = 2.19$). For the diagonal axis, the modes are polarized along the diagonals of the square: the crystal, not the shape of the core, sets their polarization. Both modes are almost linearly polarized, but not exactly: near the corners of the core the field bends, and the small $E_z$ component (not shown) gives the modes a hybrid character that no scalar model captures.

### Dispersion curves

Waveguides are characterized by how the effective index changes with frequency. We sweep $k_0 t$ from 3.5 to 15 for the three orientations and record each guided mode together with its polarization: along $x$ when the optic axis points along $z$ or $x$, along the optic axis when it is diagonal. The sweep takes a few minutes.

In [ ]:
k0ts = np.linspace(3.5, 15, 20)
sweep = {}
for name, c in axes.items():
    u = c if c[2] == 0 else [1, 0, 0]
    points = []
    for k0t in k0ts:
        neff, modes = guided_modes(c, k0t)
        points += [(k0t, n, polarization(E, u)) for n, E in zip(neff, modes)]
    sweep[name] = np.array(points)
    print(f"optic axis {name}: {len(points)} guided modes found")

Koshiba and co-workers plotted $(\beta/k_0)^2 = n_\text{eff}^2$ against $k_0 t$. Their curves (Figs. 3, 4 and 7 of the paper, digitized for the [gyptis example](https://gyptis.gitlab.io/examples/waveguides/plot_anisotropic_waveguide.html) this notebook is based on) appear as grey dots, our modes as coloured circles.

In [ ]:
REF = "https://gitlab.com/gyptis/gyptis/-/raw/master/examples/waveguides/data_aniso_fig{}.csv"
figures = {"along z": 7, "along x": 3, "diagonal": 4}


def reference(name):
    with urllib.request.urlopen(REF.format(figures[name])) as f:
        return np.loadtxt(f, delimiter=",", usecols=(0, 1)).T


fig, axs = plt.subplots(1, 3, figsize=(10, 3.8), sharey=True)
for ax, (name, points) in zip(axs, sweep.items()):
    ref = reference(name)
    ax.plot(*ref, ".", c="0.75", ms=1.5, label=f"Koshiba et al., Fig. {figures[name]}")
    sc = ax.scatter(points[:, 0], points[:, 1] ** 2, c=points[:, 2], cmap="coolwarm",
                    vmin=0, vmax=1, s=22, edgecolors="k", linewidths=0.3, zorder=3, label="gyptis")
    for eps, label in ((eps_o, r"$\varepsilon_o$"), (eps_e, r"$\varepsilon_e$")):
        ax.axhline(eps, c="0.5", lw=0.6, ls="--")
        ax.text(15.2, eps, label, va="center", fontsize="small", color="0.4")
    ax.set(xlim=(2, 15), ylim=(eps_clad, 2.32), xlabel="$k_0 t$", title=f"optic axis {name}")
axs[0].set_ylabel(r"$(\beta/k_0)^2$")
axs[0].legend(frameon=False, loc="upper left", fontsize="small", markerscale=1)
cb = fig.colorbar(sc, ax=axs, shrink=0.85, pad=0.04)
cb.set_label("fraction of $|\\mathbf{E}_t|^2$ along $x$\n(along the optic axis if diagonal)")
plt.show()

The computed modes fall on the reference curves of all three cases. Every curve starts near the cladding line $(\beta/k_0)^2 = \varepsilon_\text{clad}$, where the mode spreads far into the cladding, and rises towards the core permittivity as the frequency grows and the mode concentrates in the core. Near the cladding line our curves stop a little before the reference ones: there the mode extends beyond our box and into the PMLs, which give its effective index an imaginary part, so our filter discards it. A wider box would follow the curves further down.

The colours show the polarization. With the optic axis along $z$, the fundamental pair is degenerate: its two members (polarized along $x$ and $y$) overlap on the plot. With the optic axis along $x$ or along the diagonal, each family of modes splits into a branch polarized across the axis (blue, upper) and a branch polarized along it (red, lower), each heading towards its own bulk permittivity, $\varepsilon_o$ or $\varepsilon_e$.

### Modal birefringence

The difference between the effective indices of the two fundamental modes, the **modal birefringence**, is what sets the length over which a polarization state beats in the guide. We extract it from the sweep: at each frequency, the most confined mode polarized mostly across the axis and the most confined one polarized mostly along it.

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 3.6))
for name, marker in (("along x", "o"), ("diagonal", "s")):
    points = sweep[name]
    k, dn = [], []
    for k0t in k0ts:
        at = points[points[:, 0] == k0t]
        across, along = at[at[:, 2] < 0.5, 1], at[at[:, 2] > 0.5, 1]
        if len(across) and len(along):
            k.append(k0t)
            dn.append(across.max() - along.max())
    ax.plot(k, dn, marker + "-", mfc="none", label=f"optic axis {name}")
ax.axhline(eps_o**0.5 - eps_e**0.5, c="0.5", ls="--", lw=0.8)
ax.text(3.6, eps_o**0.5 - eps_e**0.5 - 0.0025, "bulk $n_o - n_e$", color="0.4", fontsize="small")
ax.set(xlabel="$k_0 t$", ylabel=r"$n_\mathrm{eff}^{\perp} - n_\mathrm{eff}^{\parallel}$", ylim=(0, 0.045))
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

The two in-plane orientations give the same curve, as the symmetry argument above predicts. At low frequency the modes live mostly in the isotropic cladding and the birefringence is small, about a third of the bulk value at $k_0 t \approx 5$. As the frequency grows, the modes concentrate in the core and the birefringence climbs towards the bulk value $n_o - n_e \approx 0.040$, which it reaches only for a core much larger than the wavelength. The extraordinary-like mode lags behind because it also sees a smaller core–cladding contrast, $\varepsilon_e - \varepsilon_\text{clad}$ instead of $\varepsilon_o - \varepsilon_\text{clad}$, so it spreads further into the cladding. The size of the core, relative to the wavelength, is therefore a knob on the birefringence of the guide.

### How accurate is the mesh?

We refine the mesh, keeping the ratio between cladding and core element sizes, and follow the two fundamental modes at $k_0 t = 10$ for the optic axis along $x$. As a reference we use quadratic elements on the finest mesh.

In [ ]:
k0t = 10
divisions = [4, 6, 8, 10, 14]
results = []
for n in divisions:
    g = make_geometry(t / n, 10 * t / (3 * n))
    results.append(guided_modes(axes["along x"], k0t, geom=g, n_eig=4)[0][:2])
results = np.array(results)
reference_neff = guided_modes(axes["along x"], k0t, geom=g, n_eig=4, degree=(2, 2))[0][:2]
print("reference (quadratic elements):", reference_neff.round(6))

fig, ax = plt.subplots(figsize=(4.8, 3.4))
d = np.array(divisions)
for i, label in enumerate(("across the axis", "along the axis")):
    ax.loglog(d, abs(results[:, i] - reference_neff[i]), "o-", mfc="none", label=label)
ax.loglog(d, abs(results[0, 0] - reference_neff[0]) * (d / d[0]) ** -2, "--", c="0.6", label="$\\propto h^2$")
ax.set_xticks(d, [str(x) for x in d])
ax.xaxis.set_minor_locator(plt.NullLocator())
ax.set(xlabel="elements across the core", ylabel="error on $n_\\mathrm{eff}$")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

The error on the effective index falls roughly as $h^2$, as expected for lowest-order elements on a smooth field. With ten elements across the core, as in the sweep, it is of order $10^{-4}$, well below anything visible on the dispersion curves.

### Things to try

Every cell above is editable. A few ideas:

- **Stronger anisotropy**: lithium niobate at telecom wavelengths has $n_o \approx 2.21$, $n_e \approx 2.14$. Change `eps_o`, `eps_e` and the cladding (silica, $\varepsilon \approx 2.09$), and watch the modes become much more confined.
- **Any in-plane angle**: set the optic axis to `[np.cos(a), np.sin(a), 0]` and check that the polarization of the fundamental modes follows the angle `a`.
- **Rectangular core**: make the core wider than it is thick (in `make_geometry`). Shape birefringence then adds to, or cancels, the birefringence of the crystal; find the aspect ratio where the two fundamental modes become degenerate again.
- **Near cutoff**: enlarge `L` and the box to follow the curves closer to the cladding line, and see how far the field reaches into the cladding there.